# Optimizing flights linear regression

Up until now you've been using the default hyper-parameters when building your models. In this exercise you'll use cross validation to choose an optimal (or close to optimal) set of model hyper-parameters.

The following have already been created:

- `regression` — a `LinearRegression` object
- `pipeline` — a pipeline with string indexer, one-hot encoder, vector assembler and linear regression and
- `evaluator` — a `RegressionEvaluator` object.

## Instructions

- Create a parameter grid builder.
- Add grids for with `regression.regParam` (values 0.01, 0.1, 1.0, and 10.0) and `regression.elasticNetParam` (values 0.0, 0.5, and 1.0).
- Build the grid.
- Create a cross validator, specifying five folds.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/3_GridSearch/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')
flights = flights.sample(0.25, seed=13)
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

# from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoderEstimator, VectorAssembler

indexer = StringIndexer(inputCol='org', outputCol='org_idx')
# onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
assembler = VectorAssembler(inputCols=['km', 'org_dummy'], outputCol='features')

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

regression = LinearRegression(labelCol='duration')
evaluator = RegressionEvaluator(labelCol='duration')

pipeline = Pipeline(stages=[indexer, onehot, assembler, regression])

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [ ]:
# Create parameter grid
params = ____()

# Add grids for two parameters
params = params.____(____, ____) \
               .____(____, ____)

# Build the parameter grid
params = params.____()
print('Number of models to be tested: ', len(params))

# Create cross-validator
cv = ____(estimator=____, estimatorParamMaps=____, evaluator=____, ____)